# Regularization — Ridge (L2) and Lasso (L1)

## Goal

This notebook explains and derives why regularization is needed, derives
the closed-form Ridge solution, and explains why Lasso has no equivalent
closed-form solution. The key contrast — why Ridge shrinks weights toward
zero but Lasso can eliminate them entirely — is worked out from the
mathematical structure of each penalty term.

## Why Is Regularization Needed?

Imagine a model trained with a large number of features but a small
number of samples. The data available may contain noise, and with too
little data relative to features, a model trained to directly minimize
error can end up fitting that noise instead of the real underlying
pattern — since the model has no way to distinguish genuine signal from
noise when so little data is available.

This results in a model that performs very well on the training data
(having effectively memorized it, including its noise) but performs
poorly on new, unseen data. This is **overfitting**, and it's typically
visible in the model's coefficients themselves — overfitting often
requires unusually large coefficient values, since extreme weights are
what let a model swing wildly enough to chase down noise.

Regularization counters this by adding a **penalty term** to the loss
function, based on the size of the coefficients themselves:

$$ E_{\text{regularized}} = \underbrace{\sum(y_i-\hat y_i)^2}_{\text{fit the data}} + \underbrace{\text{penalty}(B)}_{\text{discourage large weights}} $$

The model can no longer minimize error by letting any single weight grow
unchecked — doing so now costs it extra, through the penalty. This forces
a tradeoff between fitting the training data and keeping weights modest,
which tends to generalize better to unseen data.

## Ridge Regression (L2) (Closed form solution)

Ridge's penalty term is $\lambda\sum_{j} B_j^2$ — written in matrix form,
$\lambda B^TB$:

$$ E_{\text{ridge}} = (Y-XB)^T(Y-XB) + \lambda B^TB $$

Expanding (reusing the expansion from the Normal Equation notebook):

$$ E_{\text{ridge}} = Y^TY - 2Y^TXB + B^TX^TXB + \lambda B^TB $$

Differentiating with respect to $B$ (reusing $\frac{\partial}{\partial B}(B^TX^TXB)=2X^TXB$
from before, and the same identity applied to the new term,
$\frac{\partial}{\partial B}(\lambda B^TB) = 2\lambda B$):

$$ \frac{\partial E_{\text{ridge}}}{\partial B} = -2X^TY + 2X^TXB + 2\lambda B $$

Setting this to zero:

$$ X^TXB + \lambda B = X^TY $$

$$ (X^TX + \lambda I)B = X^TY $$

$$ \boxed{B = (X^TX + \lambda I)^{-1}X^TY} $$

**Why does this need a "2," structurally?** $B_j^2$ is a smooth,
differentiable function everywhere, including at $B_j=0$ — the "2" comes
directly from the power rule, the same way differentiating $x^2$ always
produces a factor of $2x$. This smoothness is exactly what makes a clean,
single closed-form solution possible for Ridge.

**Why does adding $\lambda I$ fix the invertibility problem from the
multivariable notebook?** If features are highly correlated, $X^TX$ can
have very small eigenvalues, making it nearly singular (uninvertible) or
causing huge, unstable coefficients when inverted. Adding $\lambda I$
adds a constant to every diagonal entry, directly preventing those small
eigenvalues from causing instability — $X^TX + \lambda I$ stays safely
invertible even when $X^TX$ alone wouldn't be.

**What does $\lambda$ actually control?** $\lambda$ is a hyperparameter
chosen before training (typically via cross-validation), not computed
from the data. $\lambda=0$ recovers plain linear regression exactly,
since the penalty term vanishes. Larger $\lambda$ forces weights smaller,
at the cost of fitting the training data slightly less precisely.

## Why Doesn't Ridge Set Weights to Exactly Zero? (Gradient Descent Perspective)

Differentiating Ridge's penalty gives $\frac{d}{dB_j}(\lambda B_j^2) = 2\lambda B_j$
— a force **proportional to $B_j$ itself**. As $B_j$ shrinks toward zero,
this pull shrinks too. The closer a weight gets to zero, the weaker the
remaining push toward zero becomes — so the penalty can shrink weights to
very small values, but rarely has enough remaining force to push the
last distance to exactly zero. Weights settle wherever this weakening
pull balances against the data's pull to keep that feature's contribution
alive, however small.

In [1]:
import numpy as np
class My_Ridge():
    def __init__ (self,alpha):
        self.alpha=alpha
        self.intercept_=None
        self.coef_=None

    def fit(self,x_train,y_train):
        x_train=np.insert(x_train,0,1,axis=1)
        i=np.identity(x_train.shape[1])
        i[0,0]=0
        results=np.linalg.inv(np.dot(x_train.T,x_train)+self.alpha*i).dot(x_train.T.dot(y_train))
        self.intercept_=results[0]
        self.coef_=results[1:]
    def predict(self,x_test):
        return np.dot(x_test,self.coef_) + self.intercept_

In [2]:
from sklearn.linear_model import Ridge
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
x,y=make_regression(n_samples=100,n_features=2,n_informative=4,noise=20,random_state=42)

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [9]:
my_ridge=My_Ridge(alpha=1)
sk_ridge=Ridge(alpha=1)
my_ridge.fit(x_train,y_train)
my_pred=my_ridge.predict(x_test)
sk_ridge.fit(x_train,y_train)
sk_pred=sk_ridge.predict(x_test)


print("My Ridge metrics : ")
print("MSE",mean_squared_error(y_test,my_pred))
print("MAE",mean_absolute_error(y_test,my_pred))
print("R2_SCORE",r2_score(y_test,my_pred))

print("Sklearn Ridge metrics : ")
print("MSE",mean_squared_error(y_test,sk_pred))
print("MAE",mean_absolute_error(y_test,sk_pred))
print("R2_SCORE",r2_score(y_test,sk_pred))

My Ridge metrics : 
MSE 622.4632606607115
MAE 19.40007521093191
R2_SCORE 0.9366864008286686
Sklearn Ridge metrics : 
MSE 622.4632606607121
MAE 19.400075210931913
R2_SCORE 0.9366864008286685
